In [1]:
from inference_training import Configuration, ImageDataset
from inference_training import initCudaEnvironment, createTransforms
from inference_training import drawImageAndFeatureMasks
from inference_training import exportOnnxModel, writeONNXMeta, loadONNX
from inference_training import trainModel, saveModel, loadModel
from inference_training import createModelInstance, testInference
import os

In [2]:
initCudaEnvironment(numCudaDevices=1,
                    visibleCudaDevices="0",
                    clearCudaDeviceCount=False)

# multiple models

In [3]:
# train on the GPU or on the CPU, if a GPU is not available
config = Configuration()
print("Device: " + str(config.device))

def create_model(trainDirectory, testDirectory):
    
    config.setDatasetPaths(trainPath=trainDirectory, testPath=testDirectory)
    config.setFilePrefix("")
    config.setModelName("basemodel")
    config.setInputSizes(inputWidth=250, inputHeight=250)
    config.setInputCellSize(cellSizeM=0.25, minCellSizeM=0.1, maxCellSizeM=0.5)
    config.setVersion(20250121)
    config.setModelInfo(channels=3, numClasses=2+1,  # (1 + background)
                        bboxOverlap=True, bboxPerImage=250, reuseModel=False)
    config.setEpochs(1)
    description = "Model description"
    config.setOnnxInfo(producer="Tygron", description=description)
    
    config.addLegendEntry("Background", 0, "#00000000")
    config.addLegendEntry("Label name 1", 1, "#00ffbf")
    config.addLegendEntry("Label name 2", 2, "#12d900")
    
    config.setOnnxMetaData(scoreThreshold=0.2,
                           maskThreshold=0.3,
                           strideFraction=0.5)
    
    config.setTensorInfo(tensorName='input_A:RGB_normalized', batchAmount=1)
    trainingDataset = ImageDataset(config, True, createTransforms(True))
    testDataset = ImageDataset(config, False, createTransforms(False))
    
    print("Train Image count: "+str(trainingDataset.__len__()))
    print("Test Image count: "+str(testDataset.__len__()))
    
    if not trainingDataset.validateFiles(False):
        print("Inconsistent training dataset ")
        trainingDataset.validateFiles(True)
    
    if not testDataset.validateFiles(False):
        print("Inconsistent test dataset ")
        testDataset.validateFiles(True)
    
    print("Pytorch model name " + config.getPytorchModelFileName())
    print("Onnx file name " + config.getOnnxFileName())
    
    imageNumber = 5
    print(trainingDataset.getLabelList(imageNumber))
    drawImageAndFeatureMasks(config, trainingDataset, imageNumber)
    
    model = trainModel(config, trainingDataset, testDataset)
    saveModel(config, model, path=config.getPytorchModelFileName())
    
    model.eval()
    testPrediction = testInference(config, model=model,
                               dataset=testDataset, imageNumber=88)
    
    exportOnnxModel(config, model)
    writeONNXMeta(config)
    onnx_model = loadONNX(config)
    print(f"metadata_props={onnx_model.metadata_props}")

Device: cpu


In [ ]:
base_directory = "datasets"
trainDirectory = "datasets/vogelhorst/data"
testDirectory = "datasets/landgoederenbuurt/data"

create_model(trainDirectory, testDirectory)

Train Image count: 400
Test Image count: 800
Inconsistent training dataset 
mask is missing for image datasets/vogelhorst/data\0_image.png
labels are missing for image datasets/vogelhorst/data\0_image.png
mask is missing for image datasets/vogelhorst/data\100_image.png
labels are missing for image datasets/vogelhorst/data\100_image.png
mask is missing for image datasets/vogelhorst/data\101_image.png
labels are missing for image datasets/vogelhorst/data\101_image.png
mask is missing for image datasets/vogelhorst/data\102_image.png
labels are missing for image datasets/vogelhorst/data\102_image.png
mask is missing for image datasets/vogelhorst/data\103_image.png
labels are missing for image datasets/vogelhorst/data\103_image.png
mask is missing for image datasets/vogelhorst/data\104_image.png
labels are missing for image datasets/vogelhorst/data\104_image.png
mask is missing for image datasets/vogelhorst/data\105_image.png
labels are missing for image datasets/vogelhorst/data\105_image.p

0.1%

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to C:\Users\Gebruiker/.cache\torch\hub\checkpoints\maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100.0%


Detections per image 250
in features mask: 256


C:\Users\Gebruiker\Desktop\homework\deep_learning_in_practice\engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [  0/200]  eta: 0:37:15  lr: 0.000030  loss: 2.7833 (2.7833)  loss_classifier: 1.0738 (1.0738)  loss_box_reg: 0.1110 (0.1110)  loss_mask: 1.3460 (1.3460)  loss_objectness: 0.2410 (0.2410)  loss_rpn_box_reg: 0.0114 (0.0114)  time: 11.1756  data: 0.0734
Epoch: [0]  [ 10/200]  eta: 0:35:50  lr: 0.000281  loss: 2.7833 (2.5848)  loss_classifier: 0.8470 (0.8196)  loss_box_reg: 0.1739 (0.1381)  loss_mask: 1.0034 (1.1397)  loss_objectness: 0.3122 (0.4443)  loss_rpn_box_reg: 0.0312 (0.0431)  time: 11.3193  data: 0.0455
Epoch: [0]  [ 20/200]  eta: 0:33:17  lr: 0.000532  loss: 2.0120 (2.1414)  loss_classifier: 0.3796 (0.5584)  loss_box_reg: 0.1762 (0.1612)  loss_mask: 0.7833 (0.9081)  loss_objectness: 0.3573 (0.4336)  loss_rpn_box_reg: 0.0329 (0.0801)  time: 11.0916  data: 0.0458
Epoch: [0]  [ 30/200]  eta: 0:31:21  lr: 0.000783  loss: 1.4515 (1.8560)  loss_classifier: 0.2635 (0.4803)  loss_box_reg: 0.1964 (0.1829)  loss_mask: 0.5508 (0.7723)  loss_objectness: 0.2110 (0.3481)  loss_rp

In [ ]:
#loadExistingModel = False

#if loadExistingModel:
#    model = createModelInstance(config)
#    loadModel(config, model, path=config.getPytorchModelFileName())

#else:
#    model = trainModel(config, trainingDataset, testDataset)
#    saveModel(config, model, path=config.getPytorchModelFileName())

In [ ]:
#model.eval()
#testPrediction = testInference(config, model=model,
#                               dataset=testDataset, imageNumber=88)

In [ ]:
#exportOnnxModel(config, model)
#writeONNXMeta(config)
#onnx_model = loadONNX(config)
#print(f"metadata_props={onnx_model.metadata_props}")